##### Copyright 2026 Google LLC.

In [4]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini - 멀티모달 라이브 API: 도구 사용

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_LiveAPI_tools.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

이 노트북은 [Gemini 2.5](https://ai.google.dev/gemini-api/docs/models/gemini-v2)의 멀티모달 라이브 API에서 도구를 사용하는 방법에 대한 예제를 제공합니다.

API는 Google 검색, 코드 실행, 함수 호출 도구를 제공합니다. 이전 Gemini 모델들은 이러한 도구의 버전을 지원했습니다. 라이브 API에서 Gemini 2.5의 가장 큰 변화는 기본적으로 모든 도구가 코드 실행으로 처리된다는 것입니다. 이 변화로 인해 단일 API 호출에서 **여러 도구**를 사용할 수 있으며, 모델은 단일 코드 실행 블록에서 여러 도구를 사용할 수 있습니다.

이 튜토리얼은 [이 튜토리얼](../quickstarts/Get_started_LiveAPI.ipynb)에 설명된 라이브 API에 익숙하다고 가정합니다.

## 설정

### SDK 설치

새로운 **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)**는 [Google AI for Developers](https://ai.google.dev/gemini-api/docs) 및 [Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview) API를 모두 사용하여 Gemini 2.5(및 이전 모델)에 프로그래밍 방식으로 액세스할 수 있게 해줍니다. 일부 예외를 제외하고, 한 플랫폼에서 실행되는 코드는 다른 플랫폼에서도 실행됩니다. 즉, 개발자 API를 사용하여 애플리케이션을 프로토타입화한 다음 코드를 다시 작성하지 않고 Vertex AI로 마이그레이션할 수 있습니다.

이 새로운 SDK에 대한 자세한 내용은 [문서](https://ai.google.dev/gemini-api/docs/sdks) 또는 [시작하기](../quickstarts/Get_started.ipynb) 노트북을 참조하세요.

In [5]:
%pip install -U -q google-genai

### API 키 설정

다음 셀을 실행하려면 API 키가 `GOOGLE_API_KEY`라는 이름의 Colab 시크릿에 저장되어 있어야 합니다. API 키가 없거나 Colab 시크릿을 만드는 방법을 모르는 경우, 예시는 [인증 ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb)을 참조하세요.

In [32]:
from google.colab import userdata
import os

os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')

### SDK 클라이언트 초기화

클라이언트는 환경 변수에서 API 키를 가져옵니다.
라이브 API를 사용하려면 클라이언트 버전을 `v1alpha`로 설정해야 합니다.

In [34]:
from google import genai

client = genai.Client(http_options={"api_version": "v1alpha"})

### 모델 선택

최신 안정 모델이나 미리보기 모델 중 하나를 선택합니다. `gemini-2.5-flash-native-audio-preview-09-2025`와 같은 네이티브 오디오 모델은 Colab이 요구하는 텍스트 전용 출력만 지원하지 않으므로 이 노트북에서는 작동하지 않습니다.

In [42]:
MODEL_ID = 'gemini-live-2.5-flash-preview'  # @param ['gemini-2.0-flash-live-001', 'gemini-live-2.5-flash-preview', 'gemini-2.5-flash-native-audio-preview-09-2025'] {allow-input: true, isTemplate: true}

### 임포트

In [9]:
import asyncio
import contextlib
import json
import wave

from IPython.display import display, Markdown, Audio, HTML

from google import genai
from google.genai import types

### 유틸리티

라이브 API의 오디오 출력을 사용할 것입니다. Colab에서 이를 들을 수 있는 가장 쉬운 방법은 `PCM` 데이터를 `WAV` 파일로 저장하는 것입니다:

In [10]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

디버깅 메시지를 쉽게 켜고 끌 수 있도록 로거를 사용합니다.

In [11]:
import logging
logger = logging.getLogger('Live')
logger.setLevel('INFO')
#logger.setLevel('DEBUG')  # Switch between "INFO" and "DEBUG" to toggle debug messages.

## 시작하기

라이브 API 설정의 대부분은 [시작 튜토리얼](../quickstarts/Get_started_LiveAPI.ipynb)과 유사합니다. 이 튜토리얼은 API의 실시간 상호작용에 초점을 맞추지 않으므로 코드가 단순화되었습니다. 이 코드는 라이브 API를 사용하지만, 단일 텍스트 프롬프트만 전송하고 단일 턴의 응답을 수신합니다.

어떤 예제에서든 `modality="AUDIO"`를 설정하면 음성으로 출력된 결과를 얻을 수 있습니다.

In [12]:
n = 0
async def run(prompt, modality="TEXT", tools=None):
  global n
  if tools is None:
    tools=[]

  config = {
          "tools": tools,
          "response_modalities": [modality]
  }

  async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
    display(Markdown(data=prompt))
    display(Markdown(data='-------------------------------'))
    await session.send_client_content(
      turns={"role": "user", "parts": [{"text": prompt}]}, turn_complete=True
    )

    audio = False
    filename = f'audio_{n}.wav'
    with wave_file(filename) as wf:
      async for response in session.receive():
        logger.debug(str(response))
        if response.server_content and response.server_content.model_turn and response.server_content.model_turn.parts and hasattr(response.server_content.model_turn.parts[0], 'text'):
          if text := response.server_content.model_turn.parts[0].text:
            display(Markdown(data=text))
            continue

        if response.server_content and response.server_content.model_turn and response.server_content.model_turn.parts and hasattr(response.server_content.model_turn.parts[0], 'data'):
          if data := response.server_content.model_turn.parts[0].data:
            print('.', end='')
            wf.writeframes(data)
            audio = True
            continue

        server_content = response.server_content
        if server_content is not None:
          handle_server_content(wf, server_content)
          continue

        tool_call = response.tool_call
        if tool_call is not None:
          await handle_tool_call(session, tool_call)


  if audio:
    display(Audio(filename, autoplay=True))
    n = n+1

이 튜토리얼은 여러 도구를 시연하므로 반환하는 다양한 유형의 객체를 처리하는 추가 코드가 필요합니다.

- `code_execution` 도구는 `executable_code` 및 `code_execution_result` 파트를 반환할 수 있습니다.
- `google_search` 도구는 `grounding_metadata` 객체를 첨부할 수 있습니다.

In [13]:
def handle_server_content(wf, server_content):
  model_turn = server_content.model_turn
  if model_turn:
    for part in model_turn.parts:
      executable_code = part.executable_code
      if executable_code is not None:
        display(Markdown('-------------------------------'))
        display(Markdown(f'``` python\n{executable_code.code}\n```'))
        display(Markdown('-------------------------------'))

      code_execution_result = part.code_execution_result
      if code_execution_result is not None:
        display(Markdown('-------------------------------'))
        display(Markdown(f'``` \n{code_execution_result.output}\n```'))
        display(Markdown('-------------------------------'))

  grounding_metadata = getattr(server_content, 'grounding_metadata', None)
  if grounding_metadata is not None:
    display(
        HTML(grounding_metadata.search_entry_point.rendered_content))

  return

- 마지막으로, `function_declarations` 도구를 사용하면 API가 `tool_call` 객체를 반환할 수 있습니다. 이 코드를 최소화하기 위해 `tool_call` 핸들러는 모든 함수 호출에 `"ok"` 응답으로 답합니다.

In [14]:
async def handle_tool_call(session, tool_call):
  print("Tool call:")
  function_responses = []
  for fc in tool_call.function_calls:
    function_response = types.FunctionResponse(
        id=fc.id,
        name=fc.name,
        response={"result": "ok"},
    )
    function_responses.append(function_response)
  print('>>> ', function_responses)
  await session.send_tool_response(function_responses=function_responses)

처음으로 실행해 보세요:

In [39]:
await run(prompt="Hello?", tools=None, modality = "TEXT")

Hello?

-------------------------------

Hi

 there! How can I help you today?

## 간단한 함수 호출

API의 함수 호출 기능은 다양한 함수를 처리할 수 있습니다. SDK의 지원은 아직 개발 중입니다. 따라서 간단하게 유지하여 최소한의 함수 정의인 함수 이름만 전송합니다.

라이브 API에서 함수 호출은 채팅 턴과 독립적입니다. 함수 호출이 처리되는 동안 대화가 계속될 수 있습니다.

In [43]:
turn_on_the_lights = {'name': 'turn_on_the_lights'}
turn_off_the_lights = {'name': 'turn_off_the_lights'}

In [44]:
prompt = "Turn on the lights"

tools = [
    {'function_declarations': [turn_on_the_lights, turn_off_the_lights]}
]

await run(prompt, tools=tools, modality = "TEXT")

Turn on the lights

-------------------------------

-------------------------------

``` python
print(default_api.turn_on_the_lights())
```

-------------------------------

...

-------------------------------

``` 
{'result': 'ok'}

```

-------------------------------

Turn

ed on the lights.

## 비동기 함수 호출

**비동기 함수 호출**을 사용하면 모델이 사용자 입력을 차단하지 않고 비동기적으로 함수 호출을 관리할 수 있습니다.

함수 호출이 종료될 때 모델이 아무 말도 하지 않거나, 현재 작업을 중단하거나, 현재 작업을 완료할 때까지 기다리는 등 모델의 동작 방식을 결정할 수 있습니다.

다음 셀들은 세션이 20초 동안 열린 상태를 유지하고 10초마다 모델에 전송되는 여러 요청을 수락하도록 라이브 API를 사용하는 약간 업데이트된 코드를 사용합니다. 이 구현이 궁금하다면 다음 셀을 확장하세요.

In [45]:
# @title Live class with multiple messages (just run this cell)

import collections.abc
import inspect
from asyncio.exceptions import CancelledError
import traceback

class Live:
  def __init__(self, client):
    self.client = client


  async def run(self, config, functions=None, messages=None):
    self.config = config
    self.send_queue = asyncio.Queue()
    self.tool_call_queue = asyncio.Queue()

    try:
      async with (
            client.aio.live.connect(model=MODEL_ID, config=config) as session,
            asyncio.TaskGroup() as tg
      ):
        self.session = session
        recv_task = tg.create_task(self._recv())
        send_task = tg.create_task(self._send())
        tool_call_task = tg.create_task(self._run_tool_calls(functions))
        read_text= tg.create_task(self._read_text(messages))


        await read_text
        await asyncio.sleep(20) # Keeping the socket open for 20s to wait for the FC and different messages

        raise CancelledError
    except CancelledError:
      pass
    except ExceptionGroup as EG:
      traceback.print_exception(EG)

  async def _recv(self):
    try:
      mode = None
      while True:
        async for response in self.session.receive():
          logger.debug(str(response))
          if response.server_content and response.server_content.model_turn and response.server_content.model_turn.parts and hasattr(response.server_content.model_turn.parts[0], 'text'):
            if text := response.server_content.model_turn.parts[0].text:
              if mode != 'text':
                mode = 'text'
                print()
              print(text)
          else:
            if mode == 'text':
              mode = 'other'
              print()
            print(f'<<<  {response.model_dump_json(exclude_none=True)}\n')

          tool_call = response.tool_call
          if tool_call is not None:
            await self.tool_call_queue.put(tool_call)

    except asyncio.CancelledError:
      pass

  async def _send(self):
    while True:
      msg = await self.send_queue.get()
      print(f'>>> {repr(msg)}\n')
      await self.session.send_client_content(turns=msg,turn_complete=True)

  async def _run_tool_calls(self, functions):
    while True:
      tool_call = await self.tool_call_queue.get()
      for fc in tool_call.function_calls:
        fun = functions[fc.name]
        called = fun(**fc.args)
        if inspect.iscoroutine(called):
          print(f'>> Starting {fc.name}\n')
          result = await called
          print(f'>> Done {fc.name} >>> {repr(result)}\n')
          result = self._wrap_function_result(fc, result)
          await self.session.send_tool_response(function_responses=[result])
        elif isinstance(called, collections.abc.AsyncIterable):
          async for result in called:
            result.will_continue=True
            result = self._wrap_function_result(fc, result)
            print(f">>> {repr(result)}\n")
            await self.session.send_tool_response(function_responses=[result])

          result = self._wrap_function_result(
              fc,
              types.FunctionResponse(will_continue=False)
          )
          print(f">>> {repr(result)}\n")
          await self.session.send_tool_response(
              function_responses=[result]
          )


        else:
          raise TypeError(f"expected {fc.name} to return a coroutine, or an "
                          f"AsyncIterable, got {type(fun)}")

  def _wrap_function_result(self, fc, result):
    if result is None:
      return types.FunctionResponse(
          name=fc.name,
          id=fc.id,
          response={'result': 'ok'}
      )
    elif isinstance(result, types.FunctionResponse):
      result.name = fc.name
      result.id = fc.id
      return result
    else:
      return types.FunctionResponse(
          name=fc.name,
          id=fc.id,
          response= {'result': result}
      )

  async def _read_text(self, messages):
    if messages:
        for n, message in enumerate(messages):
            await self.send_queue.put({
                'role': 'user',
                'parts': [{'text': message}]
            })
            if n+1 < len(messages):
              await asyncio.sleep(5)
    else:
        while True:
            message = await asyncio.to_thread(input, "message > ")
            if message.lower() == "q":
                break
            await self.send_queue.put({
                'role': 'user',
                'parts': [{'text': message}]
            })

### 기본 동작: 차단

기본 동작부터 시작합니다. 먼저 10초를 기다려 계산 시간을 시뮬레이션하는 모의 날씨 함수를 정의합니다.

기본 동작은 FIFO 큐처럼 작동합니다. 함수 호출이 큐에 추가되고, 이후의 모든 요청은 처리가 완료될 때까지 그 뒤에 큐에 쌓입니다(차단됩니다).

In [46]:
# Mock function, takes 10s to process
async def get_weather_vegas():
  await asyncio.sleep(10)
  return {'weather': "Sunny, 42 degrees"}

# multiple prompts, they are going to be asked with 5s delay between each of them.
questions = [
    "What's the weather in Vegas?",
    "In the meantime tell me about the Paris casino"
]

await Live(client).run(
    messages=questions,
    functions={
        'get_weather_vegas': get_weather_vegas,
    },
    config={
        "response_modalities": ["TEXT"],
        "tools": [
            {
                'function_declarations': [
                    {'name': 'get_weather_vegas',  "behavior": "UNSPECIFIED"}, # This is default behavior, equivalent to BLOCKING
                ]
            }
        ]
    }
)

>>> {'role': 'user', 'parts': [{'text': "What's the weather in Vegas?"}]}

<<<  {"tool_call":{"function_calls":[{"id":"function-call-395220346347865271","args":{},"name":"get_weather_vegas"}]}}

>> Starting get_weather_vegas

>>> {'role': 'user', 'parts': [{'text': 'In the meantime tell me about the Paris casino'}]}

>> Done get_weather_vegas >>> {'weather': 'Sunny, 42 degrees'}


The weather in Vegas
 is Sunny, 42 degrees.

<<<  {"server_content":{"generation_complete":true}}

<<<  {"server_content":{"turn_complete":true},"usage_metadata":{"prompt_token_count":229,"response_token_count":25,"total_token_count":254,"prompt_tokens_details":[{"modality":"TEXT","token_count":229}],"response_tokens_details":[{"modality":"TEXT","token_count":25}]}}


I
 can tell you that the Paris Las Vegas Casino is a beautiful hotel and casino on
 the Las Vegas Strip in Paradise, Nevada. It is owned and operated by Caesars Entertainment and has a 540-foot-tall replica of the Eiffel Tower, a two-thirds-size

보시다시피, 모델은 `get_weather_vegas` 함수를 즉시 호출했지만 두 번째 질문은 모델이 여전히 함수 호출 결과를 기다리고 있어 무시되었습니다. 함수 호출에 응답한 후에야 두 번째 질문에 답하기 시작했습니다.

### **중단(Interrupt)**: 현재 작업을 멈추고 이 결과를 처리

이번에는 `behavior`가 `NON_BLOCKING`으로 설정되어 비동기 함수 호출을 사용합니다.

이 경우 함수 호출 결과를 받았을 때 모델이 어떻게 동작할지 정의해야 합니다. 이는 함수 내부나 함수 호출을 처리하는 스크립트에서 `FunctionResponse`에 `scheduling` 값을 추가하여 관리합니다(자동 함수 호출은 사용할 수 없으므로).

이번에 `scheduling` 동작은 "**`Interrupt`**"로, 응답을 받는 즉시 모델이 현재 하던 말을 멈추고 응답을 즉시 처리합니다.

In [47]:
# Mock function, takes 10s to process
async def get_weather_vegas():
  await asyncio.sleep(10)
  return types.FunctionResponse(
      response={'weather': "Sunny, 42 degrees"},
      scheduling="INTERRUPT"
  )

# multiple prompts, they are going to be asked with 5s delay between each of them.
questions = [
    "What's the weather in Vegas?",
    "In the meantime tell me what you know about the Paris casino and all there's to do and see in it. Then continue to tell me about the Vegas casinos until I tell you to stom talking. Don't ask me, just talk non-stop"
    "Then can you tell me what's your favorite cirque du soleil show?"
]

await Live(client).run(
    messages=questions,
    functions={
        'get_weather_vegas': get_weather_vegas,
    },
    config={
        "response_modalities": ["TEXT"],
        "tools": [
            {
                'function_declarations': [
                    {'name': 'get_weather_vegas',  "behavior": "NON_BLOCKING"},
                ]
            }
        ]
    }
)

>>> {'role': 'user', 'parts': [{'text': "What's the weather in Vegas?"}]}

<<<  {"tool_call":{"function_calls":[{"id":"function-call-2548436875001823064","args":{},"name":"get_weather_vegas"}]}}

>> Starting get_weather_vegas


It
 is running. I will respond to you once I have the results.

<<<  {"server_content":{"generation_complete":true}}

<<<  {"server_content":{"turn_complete":true},"usage_metadata":{"prompt_token_count":511,"response_token_count":28,"total_token_count":539,"prompt_tokens_details":[{"modality":"TEXT","token_count":511}],"response_tokens_details":[{"modality":"TEXT","token_count":28}]}}

>>> {'role': 'user', 'parts': [{'text': "In the meantime tell me what you know about the Paris casino and all there's to do and see in it. Then continue to tell me about the Vegas casinos until I tell you to stom talking. Don't ask me, just talk non-stopThen can you tell me what's your favorite cirque du soleil show?"}]}


While
 I wait for the weather update, let me tell you abou

보시다시피, 이번에는 모델이 "`베가스 날씨 요청이 실행 중입니다. 완료되면 알려드리겠습니다`"와 같은 말로 요청을 확인한 후, 요청한 내용을 계속 처리했습니다. 그리고 함수 응답이 돌아왔을 때 하던 말을 멈추고 날씨에 대해 알려준 뒤, 이전에 이야기하던 내용을 계속했습니다.

### **유휴 상태까지 대기(When Idle)**: 현재 작업을 완료한 후 이 결과 처리

이번에도 `behavior`는 `NON_BLOCKING`으로 설정되어 비동기 함수 호출을 사용하며 `FunctionResponse`에 `scheduling` 값을 추가해야 합니다.

이번에 `scheduling` 동작은 "**`When_idle`**"로, 모델이 현재 하던 말을 **완료할 때까지 기다린** 후에야 요청한 내용을 알려줍니다.

In [48]:
import time

# Mock function, takes 6s to process
async def get_weather_vegas():
  await asyncio.sleep(6)
  return types.FunctionResponse(
      response={'weather': "Sunny, 42 degres"},
      scheduling="WHEN_IDLE"
  )

# multiple prompts, they are going to be asked with 5s delay between each of them.
questions = [
    "What's the weather in Vegas?",
    "In the meantime, without using tools, tell me what you know about the Paris casino and all there's to do and see in it. Tell me about each casino on the strip!"
]

await Live(client).run(
    messages=questions,
    functions={
        'get_weather_vegas': get_weather_vegas,
    },
    config={
        "response_modalities": ["TEXT"],
        "tools": [
            {
                'function_declarations': [
                    {'name': 'get_weather_vegas',  "behavior": "NON_BLOCKING"},
                ]
            }
        ]
    }
)

>>> {'role': 'user', 'parts': [{'text': "What's the weather in Vegas?"}]}

<<<  {"tool_call":{"function_calls":[{"id":"function-call-14628508452628440626","args":{},"name":"get_weather_vegas"}]}}

>> Starting get_weather_vegas


Got
 it. I'm fetching the weather in Vegas.

<<<  {"server_content":{"generation_complete":true}}

<<<  {"server_content":{"turn_complete":true},"usage_metadata":{"prompt_token_count":512,"response_token_count":25,"total_token_count":537,"prompt_tokens_details":[{"modality":"TEXT","token_count":512}],"response_tokens_details":[{"modality":"TEXT","token_count":25}]}}

>>> {'role': 'user', 'parts': [{'text': "In the meantime, without using tools, tell me what you know about the Paris casino and all there's to do and see in it. Tell me about each casino on the strip!"}]}


The
 Paris Las Vegas Hotel & Casino is a well-known themed resort on the Las Vegas
 Strip, recognizable by its half-scale replica of the Eiffel Tower and other Parisian landmarks.

Here's a glim

보시다시피, 이번에는 카지노에 대해 답하는 도중 함수 호출 응답을 받았음에도 불구하고(`>> Done get_weather_vegas >>> [...] response={'weather': 'Sunny, 42 degres'})` 행 참조), 현재 답변을 완료할 때까지 기다린 후에 날씨에 대해 알려주었습니다.

### 조용히 처리(Silent): 알게 된 내용을 혼자만 알고 있기

이번에도 `behavior`는 `NON_BLOCKING`으로 설정되어 비동기 함수 호출을 사용하며 `FunctionResponse`에 `scheduling` 값이 필요합니다.

이번에 `scheduling` 동작은 "**`Silent`**"로, 모델이 함수 호출이 완료되었을 때 알려주지 않지만 나중에 대화에서 해당 지식을 사용할 수 있습니다.

In [49]:
import time

# Mock function, takes 5s to process
async def get_weather_vegas():
  time.sleep(10)
  return types.FunctionResponse(
      response={'weather': "Sunny, 42 degres"},
      scheduling="SILENT"
  )

# multiple prompts, they are going to be asked with 5s delay between each of them.
questions = [
    "What's the weather in Vegas?",
    "In the meantime tell me about the Paris casino.",
    "Is the temperature over 40 degres?"
]

await Live(client).run(
    messages=questions,
    functions={
        'get_weather_vegas': get_weather_vegas,
    },
    config={
        "response_modalities": ["TEXT"],
        "tools": [
            {
                'function_declarations': [
                    {'name': 'get_weather_vegas',  "behavior": "NON_BLOCKING"},
                ]
            }
        ]
    }
)

>>> {'role': 'user', 'parts': [{'text': "What's the weather in Vegas?"}]}

<<<  {"tool_call":{"function_calls":[{"id":"function-call-8706718868914709736","args":{},"name":"get_weather_vegas"}]}}

>> Starting get_weather_vegas

>> Done get_weather_vegas >>> FunctionResponse(
  response={
    'weather': 'Sunny, 42 degres'
  },
  scheduling=<FunctionResponseScheduling.SILENT: 'SILENT'>
)


I
 am fetching the weather information for Las Vegas. Please wait a moment.

<<<  {"server_content":{"generation_complete":true}}

<<<  {"server_content":{"turn_complete":true},"usage_metadata":{"prompt_token_count":511,"response_token_count":28,"total_token_count":539,"prompt_tokens_details":[{"modality":"TEXT","token_count":511}],"response_tokens_details":[{"modality":"TEXT","token_count":28}]}}

>>> {'role': 'user', 'parts': [{'text': 'In the meantime tell me about the Paris casino.'}]}


The
 Paris Las Vegas Casino is a French-themed hotel and casino located on the Las Vegas
 Strip. It is owned and 

보시다시피, 이번에는 함수 호출이 종료되었을 때 모델이 아무 것도 하지 않았지만, 같은 것에 대해 다시 질문했을 때 새로운 함수 호출 없이 답변했습니다.

## 코드 실행

`code_execution`을 사용하면 모델이 Python 코드를 작성하고 실행할 수 있습니다. 모델이 기억만으로 풀 수 없는 수학 문제에 시도해보세요:

In [50]:
prompt="Can you compute the largest prime palindrome under 100000."

tools = [
    {'code_execution': {}}
]

await run(prompt, tools=tools, modality="TEXT")

Can you compute the largest prime palindrome under 100000.

-------------------------------

-------------------------------

``` python
import math

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def is_palindrome(n):
    return str(n) == str(n)[::-1]

largest_prime_palindrome = 0
for i in range(99999, 1, -1):
    if is_palindrome(i):
        if is_prime(i):
            largest_prime_palindrome = i
            break

print(f"The largest prime palindrome under 100000 is: {largest_prime_palindrome}")
```

-------------------------------

...

 under 100000 is 98689.



To arrive at this answer, I first defined two helper functions:
1. `is_prime(n)`: This function checks if a given number `n` is prime. It iterates from 2 up to the square root

 of `n` to find any divisors. If it finds any, the number is not prime.
2. `is_palindrome(n)`: This function checks if a given number `n` is a palindrome by converting it to a

 string and comparing it to its reverse.

Then, I iterated downwards from 99999 to 1. For each number, I first checked if it was a palindrome using `is_palindrome()`. If it was, I then checked

 if it was prime using `is_prime()`. The first number that satisfied both conditions (i.e., was both a palindrome and a prime) was the largest prime palindrome, and I stored it and broke out of the loop.

## 합성 함수 호출

합성 함수 호출은 사용자 정의 함수와 `code_execution` 도구를 결합하는 기능을 말합니다. 모델은 이것들을 더 큰 코드 블록으로 작성한 다음, 각 호출에 대한 응답을 전송할 때까지 실행을 일시 중지합니다.


In [51]:
prompt="Can write some code to loop through and print integers from 1-20, and every time you hit a multiple of 3 turn on the lights, and every time you hit a multiple of 5 turn them off?"

tools = [
    {'code_execution': {}},
    {'function_declarations': [turn_on_the_lights, turn_off_the_lights]}
]

await run(prompt, tools=tools, modality="TEXT")

Can write some code to loop through and print integers from 1-20, and every time you hit a multiple of 3 turn on the lights, and every time you hit a multiple of 5 turn them off?

-------------------------------

-------------------------------

``` python
for i in range(1, 21):
    print(i)
    if i % 3 == 0:
        print(default_api.turn_on_the_lights())
    if i % 5 == 0:
        print(default_api.turn_off_the_lights())
```

-------------------------------

...

``` 
1
2
3
{'result': 'ok'}
4
5
{'result': 'ok'}
6
{'result': 'ok'}
7
8
9
{'result': 'ok'}
10
{'result': 'ok'}
11
12
{'result': 'ok'}
13
14
15
{'result': 'ok'}
{'result': 'ok'}
16
17
18
{'result': 'ok'}
19
20
{'result': 'ok'}

```

-------------------------------

The

 code successfully looped through numbers 1 to 20. It turned on the lights for

 multiples of 3 and turned off the lights for multiples of 5, printing the corresponding API call result for each action.

## Google 검색

`google_search` 도구를 사용하면 모델이 Google 검색을 수행할 수 있습니다. 예를 들어, 학습 데이터에 포함되기에 너무 최근의 이벤트에 대해 질문해보세요.

`AUDIO` 모드에서도 검색은 실행되지만 자세한 결과는 볼 수 없습니다:

In [52]:
prompt="When the latest Brazil vs. Argentina soccer match happened and what was the final score?"

tools = [
   {'google_search': {}}
]

await run(prompt, tools=tools, modality="TEXT")

When the latest Brazil vs. Argentina soccer match happened and what was the final score?

-------------------------------

-------------------------------

``` python
print(google_search.search(queries=["latest Brazil vs. Argentina soccer match date and score", "Brazil vs Argentina last match result"]))
```

-------------------------------

...

The latest soccer match

 between Brazil and Argentina took place on **March 25, 20

25**, as part of the FIFA World Cup qualifiers. Argentina defeated Brazil with a final score of **4-1** at the Mâs Monumental stadium in Buenos Aires [1, 3, 4, 7,

 9]. The goals for Argentina were scored by Julian Alvarez, Enzo Fernández, Alexis Mac Allister, and Giuliano Simeone [7, 9]. Matheus Cunha scored for Brazil [9].

Another recent match occurred on **November 

21, 2023**, where Argentina beat Brazil 1-0 in a 2026 World Cup qualifier [5]. The lone goal was scored by Nicolás Otamendi [5].

## 멀티 도구


그러나 새 API와의 가장 큰 차이점은 요청당 1개의 도구 사용 제한이 없어졌다는 것입니다. 이전 섹션의 작업들을 조합해 보세요:

In [53]:
prompt = """\
  Hey, I need you to do three things for me.

  1. Then compute the largest prime plaindrome under 100000.
  2. Then use google search to lookup unformation about the largest earthquake in california the week of Dec 5 2024?
  3. Turn on the lights

  Thanks!
  """

tools = [
    {'google_search': {}},
    {'code_execution': {}},
    {'function_declarations': [turn_on_the_lights, turn_off_the_lights]}
]

await run(prompt, tools=tools, modality="TEXT")

  Hey, I need you to do three things for me.

  1. Then compute the largest prime plaindrome under 100000.
  2. Then use google search to lookup unformation about the largest earthquake in california the week of Dec 5 2024?
  3. Turn on the lights

  Thanks!
  

-------------------------------

-------------------------------

``` python
def is_palindrome(n):
    return str(n) == str(n)[::-1]

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

largest_prime_palindrome = 0
for i in range(99999, 1, -1):
    if is_palindrome(i) and is_prime(i):
        largest_prime_palindrome = i
        break

print(f"The largest prime palindrome under 100000 is: {largest_prime_palindrome}")
```

-------------------------------

...

Tool call:
>>>  [FunctionResponse(
  id='function-call-6911647360850859452',
  name='turn_on_the_lights',
  response={
    'result': 'ok'
  }
)]


I'

ve completed all three of your requests:

1.  The largest prime palindrome under

 100000 is **98689**.
2.  Based on the search results, a **magnitude 7.0 earthquake** struck offshore of Cape Mendocino, Northern California, on **December 5,

 2024**. This appears to be the largest earthquake in California during that week.
3.  I have turned on the lights.

## 다음 단계

- SDK에 대한 자세한 내용은 [SDK 문서](https://googleapis.github.io/python-genai/)를 참조하세요.
- 이 튜토리얼은 고수준 SDK를 사용합니다. 저수준 세부 정보에 관심이 있다면 [이 튜토리얼의 Websocket 버전](../quickstarts/websockets/Get_started_LiveAPI_tools.ipynb)을 시도해보세요.
- 이 튜토리얼은 이러한 도구의 _기본_ 사용법만 다룹니다. 더 심층적인(그리고 더 재미있는) 예제는 [검색 도구 튜토리얼](./Search_Grounding.ipynb)을 참조하세요.

또는 [Cookbook](../gemini-2/)에서 다른 Gemini 2.5 기능을 확인해보세요. 특히 이 [멀티 도구](../examples/LiveAPI_plotting_and_mapping.ipynb) 예제와 Gemini [공간 기능](../quickstarts/Spatial_understanding.ipynb)에 관한 예제를 확인하세요.